In [3]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier, export_text

In [4]:
# 1. Load Data
train_df = pd.read_csv(r"C:\Tata AI Hackathon\Tata-AI-Hackathon\dataset\train.csv")


In [5]:
print("--- 1. HUNTING FOR MISSING VALUE LEAKS ---")
# Create a dataframe where 1 = NaN and 0 = Not NaN
na_df = train_df.isna().astype(int)
na_df['Y'] = train_df['Y']
# Check correlation of "missingness" with the defect
na_corr = na_df.corr()['Y'].sort_values(ascending=False)
print("Top features where being missing (NaN) correlates with the defect:")
print(na_corr.head(10))


--- 1. HUNTING FOR MISSING VALUE LEAKS ---
Top features where being missing (NaN) correlates with the defect:
Y      1.000000
X15    0.023265
X8    -0.006163
X21   -0.006163
X10   -0.015125
X16   -0.015125
X23   -0.015125
X24   -0.015125
X25   -0.015125
X27   -0.015125
Name: Y, dtype: float64


In [6]:
print("\n--- 2. HUNTING FOR COIL ID (TIME) LEAKS ---")
defects = train_df[train_df['Y'] == 1]['CoilID']
print(f"Defect CoilIDs range from {defects.min()} to {defects.max()}")
print("Are defects clumped together? Let's look at the gaps between defect CoilIDs:")
# Calculate the gap between sequential defects
gaps = defects.sort_values().diff().dropna()
print(f"Average gap between defects: {gaps.mean():.1f}")
print(f"Smallest gap: {gaps.min()}, Largest gap: {gaps.max()}")



--- 2. HUNTING FOR COIL ID (TIME) LEAKS ---
Defect CoilIDs range from 69 to 1499
Are defects clumped together? Let's look at the gaps between defect CoilIDs:
Average gap between defects: 22.0
Smallest gap: 1.0, Largest gap: 254.0


In [7]:
print("\n--- 3. SHALLOW DECISION TREE (HARD LOGIC RULES) ---")
X_train = train_df.drop(columns=['CoilID', 'Y'])
y_train = train_df['Y']

# We fill NaNs with a massive outlier so the tree can explicitly split on "Missing Data"
X_train_filled = X_train.fillna(-99999)

# Max depth 3: Force the model to explain its logic in 3 steps or less
tree = DecisionTreeClassifier(max_depth=3, class_weight='balanced', random_state=42)
tree.fit(X_train_filled, y_train)

# Extract the exact logic rules
rules = export_text(tree, feature_names=list(X_train.columns))
print(rules)


--- 3. SHALLOW DECISION TREE (HARD LOGIC RULES) ---
|--- X13 <= 1008.03
|   |--- X18 <= 880.19
|   |   |--- X12 <= 46.69
|   |   |   |--- class: 1.0
|   |   |--- X12 >  46.69
|   |   |   |--- class: 0.0
|   |--- X18 >  880.19
|   |   |--- X35 <= 11796976.50
|   |   |   |--- class: 0.0
|   |   |--- X35 >  11796976.50
|   |   |   |--- class: 0.0
|--- X13 >  1008.03
|   |--- X14 <= 615.19
|   |   |--- X36 <= 3682.50
|   |   |   |--- class: 1.0
|   |   |--- X36 >  3682.50
|   |   |   |--- class: 0.0
|   |--- X14 >  615.19
|   |   |--- X16 <= 25.11
|   |   |   |--- class: 0.0
|   |   |--- X16 >  25.11
|   |   |   |--- class: 1.0

